# 作用域、闭包与高阶函数

学习目标：能按词法作用域追踪变量，解释闭包捕获，并通过高阶函数组织可预测的数据变换。

前置知识：函数声明与调用、函数作为值、参数传递、let/const/var、条件循环。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/08-scope-closures-and-higher-order-functions/。

1. [main.mjs](scripts/08-scope-closures-and-higher-order-functions/main.mjs)：按正文顺序运行全部正常示例。
2. [shadow-tdz.mjs](scripts/08-scope-closures-and-higher-order-functions/shadow-tdz.mjs)：内层声明的名称在整个块内遮蔽外层，初始化前读取会失败。
3. [expression-before-assignment.mjs](scripts/08-scope-closures-and-higher-order-functions/expression-before-assignment.mjs)：var 绑定提前存在，不代表它已经保存了函数表达式的结果。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/08-scope-closures-and-higher-order-functions/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 全局、模块、函数与块作用域

作用域决定某个名称在哪些代码中可见。全局环境包含语言和宿主提供的全局绑定；普通脚本顶层的 var、函数声明与全局词法声明使用不同的全局环境部分，不能一概认为顶层声明都是 globalThis 的属性。

ES 模块顶层拥有模块作用域，顶层 var 也不自动成为全局对象属性。函数的参数和局部声明属于本次函数调用的环境；let、const 还受块边界限制，var 不受普通块限制。下面只运行 .mjs 上下文，不把结果套用到浏览器普通脚本或 Node 的 CommonJS 包装环境。

内层可声明与外层同名的变量，称为遮蔽（shadowing）；离开内层后仍访问原来的外层绑定。

```javascript
const moduleTitle = "模块";
var moduleLegacy = 1;
function showScope(parameter) {
  let label = "函数";
  {
    const label = "块";
    var retained = parameter;
    console.log(label);
  }
  console.log(label, retained, moduleTitle);
}
showScope("参数");
console.log(Object.hasOwn(globalThis, "moduleTitle"), Object.hasOwn(globalThis, "moduleLegacy"));
console.log(globalThis.Math === Math);
// 输出依次为：
// 块
// 函数 参数 模块
// false false
// true
```

## 2 词法查找由定义位置决定

词法作用域（lexical scope）按代码的嵌套关系解析名称。函数在创建时关联外层环境；执行时遇到自由变量，即没有在当前函数内声明的名称，就沿关联的外层环境查找，而不是沿调用者的局部变量查找。

因此把函数传给另一个函数，并不会让它自动访问接收方的同名局部变量。参数则是在当前调用中明确传入的数据，应和外层捕获区分。

```javascript
const place = "定义位置";
function readPlace() {
  return place;
}
function invoke(reader) {
  const place = "调用位置";
  return reader();
}
console.log(invoke(readPlace));
// 输出依次为：
// 定义位置
```

## 3 声明提升与暂时性死区

声明提升（hoisting）是对执行前建立绑定等行为的常用概括，不是引擎把源码真的搬到文件顶部。在同一个函数或模块的相应作用域内，函数声明可在其书写位置之前调用；var 绑定提前初始化为 undefined，后面的赋值仍在原位置执行。

let 和 const 也在进入作用域时建立绑定，但直到声明初始化前都不能读取，这段时期是暂时性死区（temporal dead zone，TDZ）。typeof 不能绕过已存在绑定的 TDZ；只有对完全未解析到的名称才有返回 "undefined" 的特殊处理。

函数表达式是否可调用取决于变量已初始化为什么值，不能把它当成函数声明。内层同名 let 从整个块开始遮蔽外层，即使还没运行到声明行，也不会退回去读取外层变量。

```javascript
console.log(declared(3));
function declared(value) {
  return value + 1;
}
console.log(legacy);
var legacy = 5;
console.log(legacy);
const expression = function (value) {
  return value + 2;
};
console.log(expression(3), typeof neverDeclaredInThisExample);
// 输出依次为：
// 4
// undefined
// 5
// 5 undefined
```

## 4 闭包保留的是绑定

闭包（closure）是函数与它可以访问的词法环境的组合。外层函数返回后，只要仍持有返回的函数，就能通过它访问对应环境中的绑定；不是把变量在创建时拍成一张不可变快照。

下面 makeCounter 的每次调用建立独立的 count。返回的函数改变并读取各自的 count，同一计数器多次调用共享同一绑定，两个计数器彼此独立。闭包可以封装状态，但共享状态仍是一种副作用，调用顺序会影响结果。

```javascript
function makeCounter(start) {
  let count = start;
  return function () {
    count += 1;
    return count;
  };
}
const first = makeCounter(0);
const second = makeCounter(10);
console.log(first(), first(), second(), first());
let current = "旧值";
const readCurrent = () => current;
current = "新值";
console.log(readCurrent());
// 输出依次为：
// 1 2 11 3
// 新值
```

## 5 参数和返回值中的函数

接收函数作为参数，或返回函数的函数，称为高阶函数（higher-order function）。它用函数表达变化的操作，而不是把所有分支硬编码在一个流程里。

下面 makeScale 返回固定缩放因子的函数，factor 是捕获的数值，value 是每次调用的新输入。apply 接收函数和数据，明确把返回值继续交给调用者。高阶函数本身不保证纯净或无副作用，取决于传入函数的行为。

```javascript
function makeScale(factor) {
  return value => value * factor;
}
function apply(value, operation) {
  return operation(value);
}
const triple = makeScale(3);
console.log(apply(4, triple), triple(5));
// 输出依次为：
// 12 15
```

## 6 函数组合与副作用

函数组合把前一个结果作为后一个输入。本例 compose(outer, inner) 返回一个只接收单个值的同步函数，先调用 inner，再调用 outer；outer、inner 是两个变换函数。组合成立的前提是前一个返回值适合后一个参数，且调用者理解顺序。

只依赖输入并返回结果的变换较容易推断；如果函数读取可变外部状态、修改对象或执行输出，重复调用可能有额外效果。下面的组合函数只处理数值；日志收集是第二个例子中有意展示的副作用，不混入第一个计算。

```javascript
function compose(outer, inner) {
  return value => outer(inner(value));
}
const addOne = value => value + 1;
const timesTwo = value => value * 2;
console.log(compose(timesTwo, addOne)(3), compose(addOne, timesTwo)(3));
let calls = 0;
function countedDouble(value) {
  calls += 1;
  return value * 2;
}
console.log(countedDouble(3), countedDouble(3), calls);
// 输出依次为：
// 8 7
// 6 6 2
```

## 7 循环捕获与共享引用

经典 for 头部的 let 为各轮建立独立绑定，所以每个闭包记住对应一轮的变量；var 只有一个函数或模块绑定，循环结束后再调用所有闭包时，读取的是同一个最终值。这个现象不需要定时器或异步就能观察。

如果在循环外先声明一个 let，然后在循环里反复给它赋值，闭包仍共享那个绑定。const 只固定绑定，不会深复制捕获的对象；多个闭包捕获同一个对象时，对它的修改仍然共享。

```javascript
const withVar = [];
for (var i = 0; i < 3; i += 1) {
  withVar.push(() => i);
}
const withLet = [];
for (let index = 0; index < 3; index += 1) {
  withLet.push(() => index);
}
console.log(withVar[0](), withVar[1](), withVar[2]());
console.log(withLet[0](), withLet[1](), withLet[2]());
const shared = { count: 0 };
const readShared = () => shared.count;
shared.count = 9;
console.log(readShared());
// 输出依次为：
// 3 3 3
// 0 1 2
// 9
```

## 8 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

内层声明的名称在整个块内遮蔽外层，初始化前读取会失败。

```javascript
const title = "外层";
{
  console.log(typeof title);
  let title = "内层";
}
// 预期错误：ReferenceError；Cannot access 'title' before initialization
```

Step 1：独立运行 scripts/08-scope-closures-and-higher-order-functions/shadow-tdz.mjs。

```bash
node scripts/08-scope-closures-and-higher-order-functions/shadow-tdz.mjs
```

var 绑定提前存在，不代表它已经保存了函数表达式的结果。

```javascript
run();
var run = function () { return 1; };
// 预期错误：TypeError；run is not a function
```

Step 2：独立运行 scripts/08-scope-closures-and-higher-order-functions/expression-before-assignment.mjs。

```bash
node scripts/08-scope-closures-and-higher-order-functions/expression-before-assignment.mjs
```

## 本章小结

- 名称按词法嵌套关系查找，作用域不等于调用栈。
- 提前建立绑定与完成初始化是两步，typeof 也不能绕过 TDZ。
- 闭包捕获绑定；高阶函数能封装计算，也可能携带共享状态。
- 循环中的逐轮绑定能隔离计数变量，但不会自动隔离引用到的对象。

## 练习

1. 再建立一个从 100 开始的计数器，与已有计数器交错调用；核对各自独立递增，没有互相跳号。
2. 用 compose 组合“减 1”和“乘 3”，输入 4 时比较两个顺序；核对分别为 9 与 11，并解释调用顺序。
3. 把 withLet 的 index 移到循环外声明，先预测结果再运行；核对三个闭包都读到最终值，再恢复逐轮 let。
4. 修复两个独立反例，要求正常退出；说明修复的是初始化时机还是名称遮蔽，而不是笼统说“提升失效”。

## 参考与引用来源

- TC39（tc39.es）：[§9.1 环境记录、全局与模块环境](https://tc39.es/ecma262/2025/multipage/executable-code-and-execution-contexts.html#sec-environment-records)、[§9.1.2.1 词法名称查找](https://tc39.es/ecma262/2025/multipage/executable-code-and-execution-contexts.html#sec-getidentifierreference)、[§15.2–15.3 函数创建及外层环境](https://tc39.es/ecma262/2025/multipage/ecmascript-language-functions-and-classes.html#sec-function-definitions)、[§14.2.3 块声明实例化](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-blockdeclarationinstantiation)、[§14.7.4.4 逐轮绑定](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-createperiterationenvironment)、[§10.2.11 参数、arguments 和声明实例化](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-functiondeclarationinstantiation)：ECMAScript 2025 的绑定、TDZ、函数词法环境和循环捕获机制。
- MDN：[Closures](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Closures)、[Functions](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Functions)：闭包、高阶使用、循环捕获与函数作用域的教学对照。